# Wafer surface metrology and warpage characterisation

End-to-end walkthrough of the `wafer-metrology-engine` package: synthesise a 300 mm wafer, decompose its shape into Zernike terms, measure it interferometrically, report SEMI flatness, extract defects, and qualify the gauge with a Gage R&R study.

**Why these numbers matter.** In silicon-photonics packaging a fibre must land on a waveguide facet with sub-micron accuracy. Wafer shape sets how far the die surface wanders from the plane the aligner assumes: global *warp* drives chuck-to-chuck variation, *site flatness* (SFQR) drives die-level tilt and standoff, and *nanotopography* drives the residual roughness. Every micron of uncorrected shape is coupling loss.

All lengths are in **metres** unless a name says otherwise (`*_nm`, `*_um`, `*_mm`).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd

%matplotlib inline

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

N_PIXELS = 512   # drop to 256 for a faster pass
SEED = 0

## 1. Synthesise a wafer

The surface superposes four contributions: a parabolic **bow**, low-order **warp lobes** (astigmatism, coma, trefoil), smooth **nanotopography**, and discrete **defects** (particles and scratches) injected at known positions so detection can be scored.

`synthesize_wafer_pair` also builds a back surface, because the SEMI definitions need both: shape metrics live on the *median* surface `(front + back)/2`, where thickness cancels, while TTV is pure thickness.

In [ ]:
from wafer_metrology.plotting import plot_wafer_panels
from wafer_metrology.synthesize import synthesize_wafer_pair

pair = synthesize_wafer_pair(n_pixels=N_PIXELS, seed=SEED)
surface = pair.front

print(f"grid       {surface.n_pixels} x {surface.n_pixels} px  ({pair.pixel_size * 1e3:.3f} mm/px)")
print(f"PV         {surface.pv() * 1e6:.2f} um")
print(f"RMS        {surface.rms() * 1e6:.2f} um")
print(f"defects    {len(pair.defects)} injected")

fig, _ = plot_wafer_panels(
    pair.x, pair.y,
    [pair.z_front, pair.median, pair.thickness - np.nanmean(pair.thickness)],
    ["Front surface", "Median surface (shape)", "Thickness variation"],
    radius=pair.radius, shared_scale=False, suptitle="Synthetic 300 mm wafer",
)

## 2. Zernike decomposition

Zernike polynomials are orthogonal on a circular aperture, so they are the natural basis for a wafer. The basis here is **orthonormal**, which means each coefficient *is* the RMS height that term contributes — the bar chart reads directly as a shape error budget.

Flattening removes piston (an arbitrary datum), tilt (chucking) and power (the bow a scanner focuses out). What survives is the residual warp and nanotopography that genuinely eat focus budget.

In [ ]:
from wafer_metrology.zernike import (
    fit_surface, flatten, plot_coefficient_spectrum, plot_flatten_summary, term_label,
)

fit = fit_surface(surface, nmax=6)
flattened, _ = flatten(surface, nmax=6, remove_power=True)

print(f"{len(fit.terms)} terms, {100 * fit.variance_explained:.3f}% of variance explained\n")
for term, value in fit.dominant(6):
    print(f"  {term_label(term):22s} {value * 1e6:+8.3f} um RMS")
print(f"\nPV {surface.pv() * 1e6:.2f} um  ->  {flattened.pv() * 1e6:.3f} um after flattening")

fig, _ = plot_coefficient_spectrum(fit)
fig, _ = plot_flatten_summary(surface, flattened, fit)

## 3. Interferometric measurement

A reflection interferometer converts height to phase as `phi = 4*pi*z/lambda`, records phase-shifted fringes, then recovers and unwraps the phase.

**The sampling limit is the whole story.** Unwrapping only works while phase changes by less than `pi` between adjacent pixels — under `lambda/4` of height per pixel. A wafer with tens of microns of warp violates that badly at 633 nm. Real tools solve it with a **synthetic wavelength** built from two close laser lines, which extends the unambiguous range by `Lambda/lambda` at the cost of amplifying noise by the same factor. Check first, measure second.

In [ ]:
from wafer_metrology import HENE_WAVELENGTH
from wafer_metrology.interferometry import (
    check_sampling, measure_surface, plot_interferogram, plot_reconstruction,
    synthetic_wavelength,
)

LAMBDA_SYNTH = synthetic_wavelength(632.8e-9, 640.0e-9)

for name, lam in (("HeNe 633 nm", HENE_WAVELENGTH), (f"synthetic {LAMBDA_SYNTH * 1e6:.1f} um", LAMBDA_SYNTH)):
    worst, ok = check_sampling(surface.z, lam, surface.mask)
    print(f"{name:24s} worst step {worst:7.2f} rad/px   {'OK' if ok else 'ALIASED'}")

In [ ]:
measurement = measure_surface(surface, LAMBDA_SYNTH, n_steps=4, noise=0.01, seed=SEED + 11)

print(f"RMS reconstruction error  {measurement.rms_error * 1e9:.2f} nm")
print(f"PV  reconstruction error  {measurement.pv_error * 1e9:.2f} nm")

fig, _ = plot_interferogram(surface, measurement.frames, measurement.wrapped)
fig, _ = plot_reconstruction(surface, measurement)

### Single-wavelength interferometry, where it applies

Once shape is removed and no steep defects remain, a plain HeNe measurement resolves nanotopography to **sub-nanometre** — three orders of magnitude better than the synthetic-wavelength scan above. That is the trade the synthetic wavelength buys range with.

In [ ]:
from wafer_metrology.synthesize import synthesize_wafer

clean = synthesize_wafer(n_pixels=N_PIXELS, seed=SEED, n_particles=0, n_scratches=0)
smooth, _ = flatten(
    clean, nmax=6, remove_power=True,
    extra_terms=[(2, -2), (2, 2), (3, -3), (3, -1), (3, 1), (3, 3)],
)
hene = measure_surface(smooth, HENE_WAVELENGTH, noise=0.005, seed=SEED + 12)

print(f"nanotopography RMS   {smooth.rms() * 1e9:.2f} nm")
print(f"measured to          {hene.rms_error * 1e9:.3f} nm RMS error")

fig, _ = plot_reconstruction(smooth, hene, title="HeNe nanotopography scan")

## 4. SEMI flatness metrology

Now the metric set: **bow** and **warp** on the median surface, **TTV** from thickness, **sori** on the front surface, and **SFQR** per 25 x 25 mm exposure site.

SFQR is the number that gates lithography. A scanner refocuses every field, so it tracks out global shape; what it cannot track out is height variation *within* one field.

Below, the truth column comes from the synthetic wafer and the measured column from the interferometric reconstruction — the difference is pure metrology error. TTV is blank in the measured column because a reflection interferometer only sees one face.

In [ ]:
from wafer_metrology.flatness import compute_flatness, plot_flatness

truth = compute_flatness(pair, site_size=25e-3, edge_exclusion=3e-3)
reconstructed = surface.with_z(measurement.z_measured)
measured = compute_flatness(reconstructed, site_size=25e-3, edge_exclusion=3e-3)

comparison = truth.metrics.to_frame().rename(columns={"value": "truth"})
measured_series = measured.metrics.to_frame()["value"]
comparison.insert(2, "measured", measured_series.to_numpy())
comparison

In [ ]:
fig, _ = plot_flatness(truth)

print(f"complete sites: {truth.metrics.n_sites}")
truth.sites.loc[truth.sites["complete"]].nlargest(5, "sfqr_nm")[
    ["site_id", "x_center_mm", "y_center_mm", "n_points", "sfqr_nm"]
]

## 5. Defect extraction

Two independent paths, both returning the same report object.

**Classical**: flatten the shape, high-pass away the nanotopography, threshold at a robust multiple of sigma, label and measure. The background is a Gaussian-weighted *local plane* fit rather than a weighted mean — near the aperture edge the window is truncated on one side, and a mean there returns a biased background that prints a false ring of defects one window-width in from the edge.

Thresholding is **hysteretic**: seed at 5σ, grow at 2.5σ. A single hard cut chops a shallow scratch into a string of fragments wherever it dips below the threshold.

In [ ]:
from wafer_metrology.defects import detect_defects, plot_defect_map, score_detections

report = detect_defects(reconstructed)
scores = score_detections(report, pair.defects)

print(f"robust sigma  {report.sigma * 1e9:.2f} nm")
print(f"threshold     {report.threshold * 1e9:.2f} nm")
print(f"detections    {report.count}")
print(f"recall        {int(scores['detected'].sum())}/{len(scores)} injected defects\n")

fig, _ = plot_defect_map(reconstructed, report, truth=pair.defects)
report.table[["defect_id", "x_mm", "y_mm", "equiv_diameter_mm", "peak_height_nm", "kind"]]

In [ ]:
report.size_distribution()

### Unsupervised autoencoder (optional, needs PyTorch)

A small convolutional autoencoder trained **only on clean wafers** learns to reproduce smooth nanotopography and fails on anything discrete. The reconstruction error is the anomaly signal — no defect labels required, and it generalises to defect types never seen in training.

Three details decide whether this works at all:

- **The bottleneck must be narrow.** A purely convolutional stack compresses a 32×32 tile only about two-fold — ample capacity to reproduce a defect faithfully, and a defect the model reconstructs well produces no anomaly signal. A linear bottleneck of 16 numbers (64-fold compression) forces it onto the smooth manifold.
- **Train on data that went through the same measurement chain.** The reference wafers below are measured interferometrically first. A model trained on noise-free truth and scored on a noisy reconstruction sees a distribution it never learned, and flags either everything or nothing.
- **Pool the error, then threshold in log space.** A raw per-pixel squared error is a one-degree-of-freedom chi-square: hugely spread even for pure noise. Pooling over a couple of pixels and taking the log makes a robust threshold mean what it says.

In [ ]:
from wafer_metrology.defects import HAVE_TORCH

if HAVE_TORCH:
    from wafer_metrology.defects import detect_defects_ml, train_autoencoder

    # Clean reference wafers, put through the same measurement chain.
    clean_set = []
    for i in range(3):
        reference = synthesize_wafer(
            n_pixels=N_PIXELS, seed=SEED + 500 + i, n_particles=0, n_scratches=0
        )
        ref_meas = measure_surface(reference, LAMBDA_SYNTH, noise=0.01, seed=SEED + 900 + i)
        clean_set.append(reference.with_z(ref_meas.z_measured))

    model, scale = train_autoencoder(clean_set, epochs=10, seed=SEED)

    ml_report = detect_defects_ml(model, reconstructed, scale)
    ml_scores = score_detections(ml_report, pair.defects)
    print(f"autoencoder detections {ml_report.count}, "
          f"recall {int(ml_scores['detected'].sum())}/{len(ml_scores)}")

    fig, _ = plot_defect_map(reconstructed, ml_report, truth=pair.defects)
else:
    print("PyTorch not installed - skipping. pip install 'wafer-metrology-engine[ml]'")

## 6. DOE: repeatability and Gage R&R

A metrology number is worthless without knowing how much of it is the wafer and how much is the gauge. The crossed design varies:

- **part** — a distinct wafer (real variation the gauge must resolve),
- **operator** — a distinct phase-step calibration error (a bias on every measurement that operator takes),
- **trial** — a repeat with a different noise seed (pure repeatability).

Variance components follow the standard crossed ANOVA method. `%GRR` under 10 % is acceptable, over 30 % fails.

In [ ]:
from wafer_metrology.doe import (
    anova_table, gauge_rr_all, plot_doe_summary, repeatability_summary,
    run_gauge_rr, run_noise_sweep,
)

study = run_gauge_rr(n_parts=3, n_operators=3, n_trials=3, n_pixels=256, noise=0.01, seed=SEED)
sweep = run_noise_sweep(n_repeats=10, n_pixels=256, wafer_seed=SEED, seed=SEED)

print(f"{len(study)} measurements in the crossed design, {len(sweep)} in the noise sweep")
anova_table(study, "warp_um")

In [ ]:
components = gauge_rr_all(study)
components[["response", "ev", "av", "grr", "pv", "pct_grr", "ndc", "verdict"]]

Note how the verdict differs by response. **Warp** is measured well: the gauge error is a few percent of the wafer-to-wafer spread. **SFQR max** does badly — it is an extreme value over ~90 sites, so it picks up the worst noise excursion anywhere on the wafer. That is an argument for monitoring a robust statistic rather than a maximum, and it is exactly the kind of conclusion a Gage R&R study exists to produce.

In [ ]:
fig, _ = plot_doe_summary(sweep, study, components)
repeatability_summary(sweep)[
    ["noise", "n", "warp_um_std", "sfqr_mean_nm_std", "sfqr_max_nm_std"]
]

## Export for JMP

The study is already long-format — one row per measurement, factors as columns — which is what JMP's Variability/Gauge platform expects.

In [ ]:
from wafer_metrology.doe import export_for_jmp

results = Path.cwd().parent / "results"
export_for_jmp(study, results / "doe_runs_for_jmp.csv")
truth.to_csv(results / "flatness_metrics.csv")
truth.sites_to_csv(results / "site_flatness.csv")

print("wrote:")
for path in sorted(results.glob("*.csv")):
    print("  ", path.name)

## Summary

| Stage | What it produced |
|---|---|
| Synthesis | 300 mm and 150 mm wafers with injected defects; gravity-sag numbers for the mount decision |
| Zernike | shape error budget; bow is the dominant term |
| Interferometry | full-wafer shape at the synthetic wavelength; sub-nm nanotopography at 633 nm |
| Flatness | bow, warp, TTV, sori and per-site SFQR |
| Defects | every injected defect recovered, classically and unsupervised |
| DOE | %GRR and ndc per response, raw runs exported for JMP |
| Deflectometry | hardware-matched Leg A: <0.1% recovery after flat-subtraction; reversal's parity limit demonstrated |
| Capstone | measured warp priced in dB per die site — the headline sentence |

To reproduce everything as files, run `python run.py` from the project root.

In [ ]:
from wafer_metrology.deflectometry import (
    DeflectometrySetup, absolute_calibrate, fit_gamma, measure_deflectometry,
    plane_aligned_difference, plot_deflectograms, plot_deflectometry_recovery,
    plot_reversal, reversal_calibrate, rotate_surface_180, simulate_gamma_sweep,
)
from wafer_metrology.synthesize import make_flat_surface
from wafer_metrology.zernike import reconstruct

N_DEFLECT = 256
wafer150 = synthesize_wafer(n_pixels=N_DEFLECT, diameter=150e-3, seed=SEED + 1)

# Fixed instrument systematic: rotation-even screen bow + rotation-odd pose terms.
system_zmap = reconstruct(
    {(2, 0): 4e-6, (2, 2): 2e-6, (3, 1): 3e-6, (3, -3): 1.5e-6},
    wafer150.x, wafer150.y, wafer150.radius, wafer150.mask,
)
gamma_hat = fit_gamma(*simulate_gamma_sweep(2.2, noise=0.002, seed=SEED + 31))
setup = DeflectometrySetup(noise=0.005, gamma=2.2, lut_gamma=gamma_hat,
                           system_zmap=system_zmap)
print(f"display gamma calibrated: 2.2 -> {gamma_hat:.3f}")

meas = measure_deflectometry(wafer150, setup, seed=SEED + 32)
print(f"raw error (incl. systematic): RMS {meas.rms_error * 1e9:.0f} nm")

fig, _ = plot_deflectograms(wafer150, meas)
fig, _ = plot_deflectometry_recovery(wafer150, meas, title="Raw recovery (uncalibrated)")

In [ ]:
import numpy as np

# Reversal: rotate the wafer 180 deg in the mount, re-measure, decompose.
meas_180 = measure_deflectometry(rotate_surface_180(wafer150), setup, seed=SEED + 33)
wafer_est, system_est = reversal_calibrate(meas.z_measured, meas_180.z_measured, meas.mask)

# Absolute: measure the reference flat in the wafer position, subtract.
flat = make_flat_surface(N_DEFLECT, 150e-3)
flat_meas = measure_deflectometry(flat, setup, seed=SEED + 34)
z_absolute = absolute_calibrate(meas.z_measured, flat_meas.z_measured)

def aligned_rms(estimate):
    err = plane_aligned_difference(estimate, wafer150.z, wafer150.x, wafer150.y,
                                   np.isfinite(estimate))
    return float(np.sqrt(np.nanmean(err[np.isfinite(err)] ** 2)))

truth_rms = float(np.nanstd(wafer150.z[meas.mask]))
print(f"reversal wafer estimate error: RMS {aligned_rms(wafer_est) * 1e9:6.0f} nm  "
      "<- the rotation-even screen bow survives")
print(f"absolute (flat-subtracted):    RMS {aligned_rms(z_absolute) * 1e9:6.0f} nm  "
      f"= {100 * aligned_rms(z_absolute) / truth_rms:.3f}% of surface RMS "
      "(Tier-0 gate: < 1%)")

fig, _ = plot_reversal(wafer150, wafer_est, system_est, system_zmap, z_absolute)

## 8. Capstone — the measured warp priced in dB

The chain that makes this one project: the **measured** (calibrated) height map is tiled into die footprints; each site's fitted plane gives a tilt and a standoff error relative to the placement datum; those map through the coupling model into a per-site fibre-attach penalty.

Mechanisms, and what the numbers actually say:

- **Angular** — die tilt vs. the ~95 mrad divergence cone: negligible at warp scale (µdB). Worth stating, because it means angular scanning on the bench is safely cited from theory.
- **Lateral** — tilt × lever arm from the mechanical reference to the facet: the dominant term at a 2 mm lever.
- **Gap** — site standoff error perturbing the nominal working gap: second. Sites sitting *closer* than nominal show a tiny negative penalty; that is physics, not a bug.

In [ ]:
from wafer_metrology.coupling import (
    attach_loss_budget, fit_lateral_scan, lateral_loss_db, plot_loss_budget,
)

# Price the measured wafer.
measured_surface = wafer150.with_z(z_absolute)
budget = attach_loss_budget(measured_surface)   # 10 mm die, 2 mm lever, 20 µm gap
print(budget.headline, "\n")
print(budget.table[["loss_angular_db", "loss_lateral_db", "loss_gap_db",
                    "loss_total_db"]].describe().loc[["mean", "max"]])

fig, _ = plot_loss_budget(measured_surface, budget)

# And the Leg C fit the bench will produce: a ±10 µm, 1 µm-step lateral scan.
rng = np.random.default_rng(SEED)
offsets = np.arange(-10, 11, dtype=float) * 1e-6
scan = lateral_loss_db(offsets - 0.4e-6) + 0.35 + rng.normal(0, 0.05, offsets.size)
fit = fit_lateral_scan(offsets, scan)
print(f"\nlateral-scan fit: w0 = {fit.w0 * 1e6:.2f} ± {fit.w0_std * 1e6:.2f} µm "
      f"(nominal 5.20), residual {fit.residual_rms_db:.3f} dB RMS")

## Summary

| Stage | What it produced |
|---|---|
| Synthesis | 300 mm wafer, ~48 µm PV, 8 injected defects |
| Zernike | shape error budget; bow is the dominant term |
| Interferometry | full-wafer shape at the synthetic wavelength; sub-nm nanotopography at 633 nm |
| Flatness | bow, warp, TTV, sori and per-site SFQR over ~90 complete sites |
| Defects | every injected defect recovered, classically and unsupervised |
| DOE | %GRR and ndc per response, with the raw runs exported for JMP |

To reproduce everything as files, run `python run.py` from the project root.